[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/03_data_analysis_and_eda/04_data_quality_and_pitfalls.ipynb)

# Bài 04 — Chất lượng dữ liệu và các cạm bẫy khi phân tích

**🎯 Mục tiêu**
- Phát hiện và xử lý: dòng trùng lặp, category viết không thống nhất, giá trị vô lý, missing value, outlier.
- Hiểu 3 cơ chế missing: **MCAR, MAR, MNAR** — và vì sao điều đó quyết định cách xử lý.
- Nhận ra **nghịch lý Simpson** và các "tín hiệu đỏ" của **data leakage** ngay từ khâu EDA.
- Viết hàm **validate dữ liệu** tự động.

> "Garbage in, garbage out." Thực tế, phần lớn thời gian của một dự án ML là làm việc với dữ liệu — không phải chọn model.

Dữ liệu: `house_prices_dirty.csv` — phiên bản "bẩn" của dữ liệu giá nhà (giống dữ liệu crawl từ web thật).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/03_data_analysis_and_eda

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

set_style()
raw = pd.read_csv("../../data/sample/house_prices_dirty.csv")
print(raw.shape)
raw.head()

Nguyên tắc: **không sửa dữ liệu gốc**. Làm việc trên bản sao và **ghi nhật ký** (cleaning log) mọi thay đổi — để có thể giải thích và tái lập.

In [ ]:
df = raw.copy()
cleaning_log = []


def log(step, n_affected):
    cleaning_log.append({"bước": step, "số hàng bị ảnh hưởng": int(n_affected)})
    print(f"✔ {step}: {n_affected}")

## 1. Dòng trùng lặp

In [ ]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
log("xoá dòng trùng lặp hoàn toàn", n_dup)

⚠️ Trùng lặp nguy hiểm hơn bạn nghĩ: nếu một mẫu nằm ở **cả train và test**, model được "kiểm tra trên câu hỏi đã biết đáp án" → điểm test cao giả tạo (một dạng leakage). Cũng cần cẩn thận với **gần trùng lặp** (cùng căn nhà đăng 2 lần với giá hơi khác).

## 2. Category viết không thống nhất

In [ ]:
print(df["district"].value_counts())

In [ ]:
def normalize_district(s: str) -> str:
    s = s.strip().lower().replace("q.", "quận")
    return " ".join(w.capitalize() for w in s.split())


before = df["district"].copy()
df["district"] = df["district"].map(normalize_district)
log("chuẩn hoá tên quận", (before != df["district"]).sum())
print(df["district"].value_counts())

## 3. Giá trị vô lý (impossible values)

Dùng **kiến thức nghiệp vụ** để đặt ràng buộc: diện tích căn nhà 2 phòng ngủ không thể là 900 m²... Ở đây có lỗi **gõ thừa số 0** (×10). Phát hiện bằng tỉ số diện tích/số phòng ngủ:

In [ ]:
area_per_bedroom = df["area_m2"] / df["bedrooms"]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(area_per_bedroom, bins=80)
ax.set_yscale("log")
ax.set_xlabel("m² / phòng ngủ")
ax.set_title("Một cụm giá trị tách biệt rất xa → lỗi nhập liệu")
plt.show()

suspicious = area_per_bedroom > 150
print(df.loc[suspicious, ["area_m2", "bedrooms", "price_billion_vnd"]].head())

Hai lựa chọn: **sửa** (nếu chắc chắn nguyên nhân, vd chia 10) hoặc **đặt thành missing** rồi xử lý như missing value. Sửa mù quáng có thể đưa thêm lỗi; ở đây lỗi có quy luật rõ ràng (đúng ×10) nên ta sửa, và ghi log.

In [ ]:
df.loc[suspicious, "area_m2"] = df.loc[suspicious, "area_m2"] / 10
log("sửa diện tích bị gõ thừa số 0 (÷10)", suspicious.sum())

## 4. Missing values — cơ chế quan trọng hơn số lượng

In [ ]:
print(df.isna().sum())
fig, ax = plt.subplots(figsize=(8, 3))
ax.imshow(df.isna().T, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(df.shape[1]), df.columns)
ax.set_xlabel("hàng")
ax.set_title("Bản đồ missing (đen = thiếu)")
ax.grid(False)
plt.show()

| Cơ chế | Ý nghĩa | Ví dụ | Xử lý |
|---|---|---|---|
| **MCAR** (Missing Completely At Random) | thiếu hoàn toàn ngẫu nhiên | lỗi đồng bộ dữ liệu ngẫu nhiên | xoá hàng / điền đơn giản đều an toàn (chỉ mất thông tin) |
| **MAR** (Missing At Random) | việc thiếu phụ thuộc vào **biến quan sát được khác** | tuổi nhà hay bị thiếu ở quận mới | điền dựa trên các biến khác (theo nhóm, model) |
| **MNAR** (Missing Not At Random) | việc thiếu phụ thuộc vào **chính giá trị bị thiếu** | người thu nhập cao không khai thu nhập | khó nhất; thêm cột **"is_missing"** làm feature, cần hiểu nghiệp vụ |

Kiểm tra nhanh: việc thiếu có liên hệ với biến mục tiêu hay biến khác không?

In [ ]:
for col in ["age_years", "bathrooms", "distance_to_center_km"]:
    miss = df[col].isna()
    print(f"{col:22s} giá TB khi thiếu = {df.loc[miss, 'price_billion_vnd'].mean():.2f} | "
          f"khi không thiếu = {df.loc[~miss, 'price_billion_vnd'].mean():.2f}")

Ở dataset này, giá trung bình gần như không khác → phù hợp với MCAR (đúng là cách dữ liệu được sinh ra). Nếu khác biệt lớn, bản thân việc "bị thiếu" là **thông tin** — nên thêm cột chỉ báo `col_is_missing`.

Cách điền (imputation) **đúng** — học từ train rồi áp dụng cho test — được học ở chương 04. Ở đây chỉ đánh dấu:

In [ ]:
for col in ["age_years", "bathrooms", "distance_to_center_km"]:
    df[f"{col}_is_missing"] = df[col].isna().astype(int)
log("thêm cột chỉ báo missing (chưa điền — để pipeline xử lý)", df[["age_years", "bathrooms", "distance_to_center_km"]].isna().any(axis=1).sum())

## 5. Outlier — phát hiện ≠ xoá

Hai quy tắc phổ biến:
- **IQR:** ngoài $[Q_1 - 1.5\,IQR,\; Q_3 + 1.5\,IQR]$.
- **Robust z-score (MAD):** $z = 0.6745\,\frac{x - \text{median}}{\text{MAD}}$, ngưỡng $|z| > 3.5$. Không bị chính outlier làm sai lệch như z-score thường (dùng mean/std).

In [ ]:
def iqr_outliers(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return (s < q1 - k * (q3 - q1)) | (s > q3 + k * (q3 - q1))


def robust_z_outliers(s, thresh=3.5):
    mad = np.median(np.abs(s - s.median()))
    return np.abs(0.6745 * (s - s.median()) / mad) > thresh


price = df["price_billion_vnd"]
print("IQR outliers trên giá      :", iqr_outliers(price).sum())
print("IQR outliers trên log(giá) :", iqr_outliers(np.log(price)).sum())
print("Robust-z trên log(giá)     :", robust_z_outliers(np.log(price)).sum())

Trên thang gốc, giá lệch phải nên rất nhiều căn bị coi là "outlier"; trên thang log chỉ còn vài căn. **Bài học:** outlier thống kê ≠ lỗi dữ liệu. Chỉ xoá khi chắc chắn là lỗi (như mục 3). Với outlier thật, cân nhắc: log-transform, model bền vững với outlier (tree, Huber loss), hoặc cắt ngưỡng (winsorize).

## 6. Nghịch lý Simpson

Xu hướng thấy ở dữ liệu **tổng hợp** có thể **đảo ngược** khi chia theo nhóm. Ví dụ kinh điển (số liệu sỏi thận, Charig et al. 1986): hai phương pháp điều trị A và B.

In [ ]:
kidney = pd.DataFrame({
    "phương_pháp": ["A", "A", "B", "B"],
    "kích_thước_sỏi": ["nhỏ", "lớn", "nhỏ", "lớn"],
    "thành_công": [81, 192, 234, 55],
    "tổng": [87, 263, 270, 80],
})
kidney["tỉ_lệ"] = (kidney["thành_công"] / kidney["tổng"]).round(3)
print(kidney, "\n")
overall = kidney.groupby("phương_pháp")[["thành_công", "tổng"]].sum()
print("Tổng hợp:\n", (overall["thành_công"] / overall["tổng"]).round(3))

- Theo **từng nhóm** (sỏi nhỏ, sỏi lớn), **A tốt hơn B**.
- Khi **gộp lại**, **B lại "tốt hơn" A**!

Nguyên nhân: A được dùng chủ yếu cho ca **khó** (sỏi lớn, tỉ lệ thành công thấp hơn), B cho ca **dễ**. Kích thước sỏi là **biến gây nhiễu (confounder)** ảnh hưởng tới cả việc chọn phương pháp lẫn kết quả. Bài học: luôn hỏi "có biến nào khác giải thích sự khác biệt không?" — chương 38 (Causal ML) đi sâu vào vấn đề này.

## 7. Tín hiệu đỏ của data leakage trong EDA

Leakage = model được dùng thông tin **không có sẵn tại thời điểm dự đoán** → điểm offline rất cao, triển khai thực tế thất bại. Dấu hiệu cảnh báo khi EDA:

1. **Một feature dự đoán target "quá tốt"** (tương quan ≈ 1, hay một feature riêng lẻ cho AUC ≈ 0.99).
2. Feature được ghi nhận **sau** sự kiện cần dự đoán: `refund_issued` khi dự đoán gian lận, `discharge_date` khi dự đoán nhập viện, `cancellation_reason` khi dự đoán churn.
3. Feature là **phiên bản biến đổi của target**: `price_per_m2` khi dự đoán `price` (ta vừa tạo ở bài 03 — chỉ dùng cho phân tích, **không** dùng làm feature!).
4. ID hoặc thời gian vô tình mang thông tin nhãn (vd: ID tăng dần và dữ liệu sắp theo nhãn).

In [ ]:
check = df.copy()
check["price_per_m2"] = check["price_billion_vnd"] / check["area_m2"]   # ⚠️ suy ra từ target
check["refund_issued"] = (check["price_billion_vnd"] > check["price_billion_vnd"].quantile(0.9)).astype(int)  # giả lập feature "tương lai"
corr_with_target = check.select_dtypes("number").corr()["price_billion_vnd"].drop("price_billion_vnd")
print(corr_with_target.abs().sort_values(ascending=False).round(3).head(5))
print("\n→ Feature nào tương quan bất thường cao với target cần được hỏi: 'giá trị này có biết được TRƯỚC khi dự đoán không?'")

Chương 05 có hẳn một bài về data leakage.

## 8. Validate dữ liệu tự động

Đóng gói các kiểm tra thành hàm — chạy lại mỗi khi có dữ liệu mới (trong production, các công cụ như *Great Expectations*, *pandera* làm việc này):

In [ ]:
def validate(df: pd.DataFrame) -> list[str]:
    errors = []
    if df.duplicated().any():
        errors.append(f"{df.duplicated().sum()} dòng trùng lặp")
    allowed = {"Quận 1", "Quận 3", "Quận 7", "Bình Thạnh", "Thủ Đức"}
    bad = set(df["district"].dropna()) - allowed
    if bad:
        errors.append(f"quận không hợp lệ: {sorted(bad)}")
    if (df["area_m2"] / df["bedrooms"] > 150).any():
        errors.append("diện tích/phòng ngủ > 150 m² (nghi lỗi nhập liệu)")
    if (df["price_billion_vnd"] <= 0).any():
        errors.append("giá <= 0")
    for col in ["bedrooms", "area_m2", "price_billion_vnd"]:
        if df[col].isna().any():
            errors.append(f"cột bắt buộc '{col}' bị thiếu")
    return errors


print("Dữ liệu gốc :", validate(raw))
print("Sau làm sạch:", validate(df) or "✅ hợp lệ")

In [ ]:
pd.DataFrame(cleaning_log)

## 🧠 Kiểm tra nhanh

1. Vì sao dòng trùng lặp giữa train và test làm điểm đánh giá cao giả tạo?
2. Người có thu nhập cao thường bỏ trống câu hỏi thu nhập. Đây là MCAR, MAR hay MNAR? Nên xử lý thế nào?
3. Một điểm nằm ngoài râu boxplot — có nên xoá ngay không?
4. Giải thích nghịch lý Simpson trong ví dụ sỏi thận bằng một câu.
5. Nêu 2 dấu hiệu cho thấy một feature có thể gây leakage.

## 📌 Tóm tắt

- Làm việc trên bản sao + ghi cleaning log.
- Trùng lặp, category không thống nhất, giá trị vô lý: dùng kiến thức nghiệp vụ để phát hiện.
- Missing: hiểu cơ chế (MCAR/MAR/MNAR); thêm cột chỉ báo khi việc thiếu mang thông tin.
- Outlier thống kê ≠ lỗi; xem xét trên thang log; chỉ xoá khi chắc chắn là lỗi.
- Simpson: xu hướng tổng hợp có thể đảo ngược theo nhóm — tìm biến gây nhiễu.
- Leakage: nghi ngờ mọi feature "quá tốt" hoặc được ghi nhận sau thời điểm dự đoán.

## 📚 Tài liệu tham khảo

- Kaufman et al. — *Leakage in Data Mining: Formulation, Detection, and Avoidance* (KDD 2011)
- Rubin (1976) — *Inference and Missing Data* (định nghĩa MCAR/MAR/MNAR)
- Charig et al. (1986) — dữ liệu sỏi thận, ví dụ kinh điển của nghịch lý Simpson